# Tập 2.8 · YOLO: robot nhận ra người và đồ vật 🎯

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap28/tap28_yolo.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 2 · ⏱ ~45 phút · Giáo trình: Principles of Robot Autonomy, [Chương 10](https://porabook.com/book/object-detection/)

Mạng YOLO tí hon của video (đã học sẵn, `sim/trong_so.pt`) nhận ra người, hộp, bóng. Trong notebook này bạn:

1. Nạp mạng và 500 ảnh thử của video (20–60 giây).
2. **Tự viết** IoU, NMS và AP / mAP (3 bài), có ô chấm tự động.
3. Vẽ khung trước/sau NMS và đường precision-recall bằng code của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap28")
sys.path[:0] = ["bai_tap", "sim", ".."]

from kiem_tra import goc
g = goc()                          # nạp ../sim/yolo.py: mạng đã học + 500 ảnh thử
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"{g.SO_THAM_SO} tham số · ảnh thử có {sum(len(v) for v in g.V_THU)} vật")
print(f"khung trước NMS: {sum(map(len, g.THO))} · sau NMS: {sum(map(len, g.SACH))}")
print(f"AP từng lớp {dict(zip(g.LOP, [round(a * 100, 1) for a in g.AP]))} · mAP = {g.MAP * 100:.1f}% (không NMS: {g.AP_THO * 100:.1f}%)")

## 2. Bài tập

Khung dạng `(cx, cy, w, h)` theo tỉ lệ ảnh; dự đoán `(lop, diem, cx, cy, w, h)`; vật thật `(lop, cx, cy, w, h)`.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `iou(a, b)` | giao / hợp của hai khung |
| **2** | `nms(du_doan, nguong)` | xếp theo điểm, bỏ khung cùng lớp chồng ≥ ngưỡng lên khung đã giữ |
| **3** | `ap_lop(du_doan, that, c, nguong_iou)` | AP của lớp c, trả về `(AP, recall, precision)` |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def iou(a, b):
    """Bài 1: IoU = diện tích phần giao / diện tích phần hợp của hai khung a, b dạng (cx, cy, w, h).

    Đổi về góc: x0 = cx − w/2, x1 = cx + w/2 (tương tự y). Giao = max(0, min(x1) − max(x0)) · max(0, min(y1) − max(y0)).
    Hợp = w_a·h_a + w_b·h_b − giao. Trả về giao / (hợp + 1e-9)."""
    raise NotImplementedError("TODO Bài 1: IoU")


def nms(du_doan, nguong=0.5):
    """Bài 2: non-maximum suppression. YOLO thường đoán vài khung chồng lên cùng một vật; chỉ giữ khung tốt nhất.

    1. Xếp các dự đoán theo điểm giảm dần.
    2. Duyệt lần lượt: giữ dự đoán d nếu với MỌI khung g đã giữ, hoặc khác lớp, hoặc iou(khung d, khung g) < nguong.
    Trả về list các dự đoán được giữ (theo thứ tự đã xếp)."""
    raise NotImplementedError("TODO Bài 2: NMS")


def ap_lop(du_doan, that, c, nguong_iou=0.5):
    """Bài 3: AP (average precision) của lớp c trên nhiều ảnh. du_doan[k], that[k] là list của ảnh thứ k.

    1. Gom mọi dự đoán lớp c của mọi ảnh, xếp theo điểm giảm dần. n_that = số vật thật lớp c.
    2. Duyệt từng dự đoán (ảnh k): tìm vật thật lớp c CHƯA được khớp trong ảnh k có IoU lớn nhất;
       nếu IoU ≥ nguong_iou thì là đúng (TP, đánh dấu vật đó đã khớp), ngược lại là sai (FP).
    3. Cộng dồn: recall = TP_dồn / n_that, precision = TP_dồn / (TP_dồn + FP_dồn).
    4. Làm precision không tăng từ phải sang trái: p_mono = np.maximum.accumulate(precision[::-1])[::-1].
    5. AP = Σ (recall_i − recall_{i−1}) · p_mono_i   (recall_{−1} = 0).
    Trả về (AP, mảng recall, mảng precision)."""
    raise NotImplementedError("TODO Bài 3: AP")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Đổi ngưỡng NMS và ngưỡng IoU khi chấm, xem mAP thay đổi thế nào.

In [ ]:
import loi_giai
tho = [[tuple(d) for d in dd] for dd in g.THO]
for nguong_nms in [0.3, 0.5, 0.7]:
    sach = [loi_giai.nms(d, nguong_nms) for d in tho]
    for nguong_iou in [0.5, 0.75]:
        m = sum(loi_giai.ap_lop(sach, g.V_THU, c, nguong_iou)[0] for c in range(3)) / 3
        print(f"NMS {nguong_nms} · chấm IoU ≥ {nguong_iou}: mAP {m * 100:.1f}%")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap28).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)